In [1]:
import pandas as pd
import numpy as np

sales = pd.read_csv('sales_data-2.csv')

inventory = pd.read_csv('inventory_data.csv')

purchase = pd.read_csv('purchase_orders_data.csv')

suppliers = pd.read_csv('suppliers_data.csv')

FileNotFoundError: [Errno 2] No such file or directory: 'sales_data-2.csv'

In [2]:
from google.colab import files

uploaded = files.upload()

Saving inventory_data.csv to inventory_data.csv
Saving purchase_orders_data.csv to purchase_orders_data.csv
Saving sales_data-2.csv to sales_data-2.csv
Saving suppliers_data.csv to suppliers_data.csv


In [3]:
import pandas as pd
import numpy as np

sales = pd.read_csv('sales_data-2.csv')

inventory = pd.read_csv('inventory_data.csv')

purchase = pd.read_csv('purchase_orders_data.csv')

suppliers = pd.read_csv('suppliers_data.csv')

In [4]:
sales.head()

inventory.head()

purchase.head()

suppliers.head()

sales.info()

inventory.info()

purchase.info()

suppliers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Sale_ID        5000 non-null   object
 1   Product_ID     5000 non-null   object
 2   Store_ID       5000 non-null   object
 3   Sale_Date      5000 non-null   object
 4   Quantity_Sold  5000 non-null   int64 
 5   Revenue        5000 non-null   int64 
dtypes: int64(2), object(4)
memory usage: 234.5+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Product_ID     200 non-null    object
 1   Store_ID       200 non-null    object
 2   Warehouse_ID   200 non-null    object
 3   Stock_Level    200 non-null    int64 
 4   Reorder_Level  200 non-null    int64 
 5   Last_Updated   200 non-null    object
dtypes: int64(2), object(4)
memory usage: 9.5+ KB


In [5]:
sales.isnull().sum()

inventory.isnull().sum()

purchase.isnull().sum()

suppliers.isnull().sum()

,0
Supplier_ID,0
Supplier_Name,0
Product_ID,0
Lead_Time (days),0
Order_Frequency,0


In [6]:
sales = sales.drop_duplicates()

inventory = inventory.drop_duplicates()

purchase = purchase.drop_duplicates()

suppliers = suppliers.drop_duplicates()

In [7]:
sales['Sale_Date'] = pd.to_datetime(
sales['Sale_Date']
)

inventory['Last_Updated'] = pd.to_datetime(
inventory['Last_Updated']
)

purchase['Order_Date'] = pd.to_datetime(
purchase['Order_Date']
)

purchase['Arrival_Date'] = pd.to_datetime(
purchase['Arrival_Date']
)

In [8]:
purchase['Lead_Time'] = (
purchase['Arrival_Date']
-
purchase['Order_Date']
).dt.days

In [9]:
dim_product = pd.DataFrame(
sales['Product_ID'].unique(),
columns=['Product_ID']
)

dim_product['Product_Key'] = range(
1,
len(dim_product)+1
)

In [10]:
dim_store = pd.DataFrame(
sales['Store_ID'].unique(),
columns=['Store_ID']
)

dim_store['Store_Key'] = range(
1,
len(dim_store)+1
)

In [11]:
dim_supplier = suppliers.copy()

dim_supplier['Supplier_Key'] = range(
1,
len(dim_supplier)+1
)

In [12]:
dim_warehouse = pd.DataFrame(
inventory['Warehouse_ID'].unique(),
columns=['Warehouse_ID']
)

dim_warehouse['Warehouse_Key'] = range(
1,
len(dim_warehouse)+1
)

In [13]:
dim_date = pd.DataFrame()

dim_date['Full_Date'] = pd.date_range(
sales['Sale_Date'].min(),
sales['Sale_Date'].max()
)

dim_date['Date_Key'] = dim_date[
'Full_Date'
].dt.strftime('%Y%m%d').astype(int)

dim_date['Day'] = dim_date['Full_Date'].dt.day

dim_date['Month'] = dim_date['Full_Date'].dt.month

dim_date['Quarter'] = dim_date['Full_Date'].dt.quarter

dim_date['Year'] = dim_date['Full_Date'].dt.year

In [14]:
fact_supply_chain = sales.merge(
inventory,
on=['Product_ID','Store_ID'],
how='left'
)

fact_supply_chain = fact_supply_chain.merge(
purchase[
['Product_ID',
'Supplier_ID',
'Quantity',
'Lead_Time']
],
on='Product_ID',
how='left'
)

fact_supply_chain.rename(
columns={
'Quantity':'Purchase_Quantity'
},
inplace=True
)

fact_supply_chain = fact_supply_chain.fillna(0)

In [15]:
dim_product.to_csv(
'dim_product.csv',
index=False
)

dim_store.to_csv(
'dim_store.csv',
index=False
)

dim_supplier.to_csv(
'dim_supplier.csv',
index=False
)

dim_warehouse.to_csv(
'dim_warehouse.csv',
index=False
)

dim_date.to_csv(
'dim_date.csv',
index=False
)

fact_supply_chain.to_csv(
'fact_supply_chain.csv',
index=False
)

In [16]:
from google.colab import files

files.download('dim_product.csv')
files.download('dim_store.csv')
files.download('dim_supplier.csv')
files.download('dim_warehouse.csv')
files.download('dim_date.csv')
files.download('fact_supply_chain.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [17]:
print(dim_date.shape)
print(fact_supply_chain.shape)

(5000, 6)
(10013344, 13)


In [18]:
import os

print(os.path.getsize('dim_date.csv'))
print(os.path.getsize('fact_supply_chain.csv'))

159754
760725402


In [19]:
fact_supply_chain[['Product_ID']].drop_duplicates()


,Product_ID
0,P001
1358,P003
7126,P004
16490,P005
20156,P002


In [20]:
fact_supply_chain.shape

(10013344, 13)

In [21]:
fact_supply_chain = fact_supply_chain.fillna(0)

fact_supply_chain.to_csv(
    'fact_supply_chain.csv',
    index=False,
    encoding='utf-8'
)

In [22]:
from google.colab import files

files.download('fact_supply_chain.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [23]:
import os

size = os.path.getsize('fact_supply_chain.csv')

print(size/1024/1024)

725.4842777252197


In [24]:
purchase_summary = purchase.groupby(
    'Product_ID'
).agg({
    'Quantity':'sum',
    'Lead_Time':'mean',
    'Supplier_ID':'first'
}).reset_index()

In [25]:
fact_supply_chain = sales.merge(
    inventory,
    on=['Product_ID','Store_ID'],
    how='left'
)

In [26]:
fact_supply_chain = fact_supply_chain.merge(
    purchase_summary,
    on='Product_ID',
    how='left'
)

In [27]:
fact_supply_chain.rename(
    columns={'Quantity':'Purchase_Quantity'},
    inplace=True
)

In [28]:
fact_supply_chain = fact_supply_chain.fillna(0)

In [29]:
fact_supply_chain.shape

(49881, 13)

In [30]:
fact_supply_chain.head()


,Sale_ID,Product_ID,Store_ID,Sale_Date,Quantity_Sold,Revenue,Warehouse_ID,Stock_Level,Reorder_Level,Last_Updated,Purchase_Quantity,Lead_Time,Supplier_ID
0,0d12b607,P001,S102,2024-01-01,322,2556,W003,389,131,2024-03-16,51827,5.0,SUP002
1,0d12b607,P001,S102,2024-01-01,322,2556,W001,59,154,2024-04-03,51827,5.0,SUP002
2,0d12b607,P001,S102,2024-01-01,322,2556,W002,74,173,2024-04-19,51827,5.0,SUP002
3,0d12b607,P001,S102,2024-01-01,322,2556,W001,418,105,2024-06-16,51827,5.0,SUP002
4,0d12b607,P001,S102,2024-01-01,322,2556,W001,183,183,2024-08-09,51827,5.0,SUP002


In [31]:
fact_supply_chain.to_csv(
    'fact_supply_chain.csv',
    index=False
)

In [32]:
from google.colab import files

files.download('fact_supply_chain.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [33]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

sales = sales.sort_values('Sale_Date')

sales['Day_Number'] = range(len(sales))

X = sales[['Day_Number']]
y = sales['Quantity_Sold']

X_train,X_test,y_train,y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

model = RandomForestRegressor()

model.fit(X_train,y_train)

RandomForestRegressor()

In [34]:
daily_demand = sales.groupby(
    'Product_ID'
)['Quantity_Sold'].mean().reset_index()

daily_demand.columns = [
    'Product_ID',
    'Avg_Daily_Demand'
]

inventory = inventory.merge(
    daily_demand,
    on='Product_ID'
)

inventory = inventory.merge(
    suppliers[['Product_ID','Lead_Time']],
    on='Product_ID'
)

inventory['Reorder_Point'] = (
    inventory['Avg_Daily_Demand']
    *
    inventory['Lead_Time']
)

KeyError: "['Lead_Time'] not in index"

In [35]:
print(suppliers.columns)

Index(['Supplier_ID', 'Supplier_Name', 'Product_ID', 'Lead_Time (days)',
       'Order_Frequency'],
      dtype='object')


In [36]:
suppliers = suppliers.rename(
    columns={'Lead_Time (days)': 'Lead_Time'}
)


In [37]:
print(suppliers.columns)

Index(['Supplier_ID', 'Supplier_Name', 'Product_ID', 'Lead_Time',
       'Order_Frequency'],
      dtype='object')


In [38]:
inventory = inventory.merge(
    suppliers[['Product_ID', 'Lead_Time']],
    on='Product_ID'
)

In [39]:
inventory['Reorder_Point'] = (
    inventory['Avg_Daily_Demand']
    *
    inventory['Lead_Time']
)

In [40]:
daily_demand = sales.groupby(
    'Product_ID'
)['Quantity_Sold'].mean().reset_index()

daily_demand.columns = [
    'Product_ID',
    'Avg_Daily_Demand'
]

inventory = inventory.merge(
    daily_demand,
    on='Product_ID'
)

inventory = inventory.merge(
    suppliers[['Product_ID','Lead_Time']],
    on='Product_ID'
)

inventory['Reorder_Point'] = (
    inventory['Avg_Daily_Demand']
    *
    inventory['Lead_Time']
)

KeyError: 'Avg_Daily_Demand'

In [41]:
inventory['Recommended_Order'] = (
    inventory['Reorder_Point']
    -
    inventory['Stock_Level']
)

inventory[
    inventory['Recommended_Order'] > 0
]

,Product_ID,Store_ID,Warehouse_ID,Stock_Level,Reorder_Level,Last_Updated,Avg_Daily_Demand_x,Lead_Time_x,Reorder_Point,Avg_Daily_Demand_y,Lead_Time_y,Recommended_Order
0,P005,S101,W003,498,163,2024-02-01,261.864337,9,2356.779034,261.864337,9,1858.779034
1,P005,S101,W003,498,163,2024-02-01,261.864337,9,2356.779034,261.864337,4,1858.779034
2,P005,S101,W003,498,163,2024-02-01,261.864337,9,2356.779034,261.864337,8,1858.779034
3,P005,S101,W003,498,163,2024-02-01,261.864337,9,2356.779034,261.864337,7,1858.779034
4,P005,S101,W003,498,163,2024-02-01,261.864337,9,2356.779034,261.864337,5,1858.779034
...,...,...,...,...,...,...,...,...,...,...,...,...
20827,P001,S104,W003,140,184,2024-08-18,249.325670,4,997.302682,249.325670,9,857.302682
20828,P001,S104,W003,140,184,2024-08-18,249.325670,4,997.302682,249.325670,9,857.302682
20829,P001,S104,W003,140,184,2024-08-18,249.325670,4,997.302682,249.325670,8,857.302682
20830,P001,S104,W003,140,184,2024-08-18,249.325670,4,997.302682,249.325670,7,857.302682


In [42]:
daily_demand = sales.groupby(
    'Product_ID'
)['Quantity_Sold'].mean().reset_index()

daily_demand.columns = [
    'Product_ID',
    'Avg_Daily_Demand'
]

In [43]:
inventory = inventory.merge(
    daily_demand,
    on='Product_ID'
)

In [44]:
suppliers.rename(
    columns={'Lead_Time (days)':'Lead_Time'},
    inplace=True
)

In [45]:
inventory = inventory.merge(
    suppliers[['Product_ID','Lead_Time']],
    on='Product_ID'
)

In [46]:
inventory['Reorder_Point'] = (
    inventory['Avg_Daily_Demand']
    *
    inventory['Lead_Time']
)

In [47]:
inventory[
[
'Product_ID',
'Avg_Daily_Demand',
'Lead_Time',
'Reorder_Point'
]
].head(20)

,Product_ID,Avg_Daily_Demand,Lead_Time,Reorder_Point
0,P005,261.864337,9,2356.779034
1,P005,261.864337,4,1047.457348
2,P005,261.864337,8,2094.914697
3,P005,261.864337,7,1833.050360
4,P005,261.864337,5,1309.321686
5,P005,261.864337,5,1309.321686
6,P005,261.864337,6,1571.186023
7,P005,261.864337,9,2356.779034
8,P005,261.864337,3,785.593011
9,P005,261.864337,3,785.593011


In [48]:
from sklearn.cluster import KMeans

X = suppliers[
['Lead_Time',
 'Order_Frequency']
]

kmeans = KMeans(
    n_clusters=3,
    random_state=42
)

suppliers['Cluster'] = kmeans.fit_predict(X)

ValueError: could not convert string to float: 'Weekly'

In [49]:
suppliers.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Supplier_ID      50 non-null     object
 1   Supplier_Name    50 non-null     object
 2   Product_ID       50 non-null     object
 3   Lead_Time        50 non-null     int64 
 4   Order_Frequency  50 non-null     object
dtypes: int64(1), object(4)
memory usage: 2.1+ KB


In [50]:
print(X.dtypes)

Lead_Time           int64
Order_Frequency    object
dtype: object


In [51]:
suppliers['Order_Frequency'] = pd.to_numeric(
    suppliers['Order_Frequency'],
    errors='coerce'
)

In [52]:
suppliers = suppliers.fillna(0)

In [53]:
from sklearn.cluster import KMeans

X = suppliers[
['Lead_Time',
 'Order_Frequency']
]

kmeans = KMeans(
    n_clusters=3,
    random_state=42
)

suppliers['Cluster'] = kmeans.fit_predict(X)

suppliers.head()

,Supplier_ID,Supplier_Name,Product_ID,Lead_Time,Order_Frequency,Cluster
0,SUP001,Murray-Ramirez,P002,9,0.0,1
1,SUP002,"Thomas, Meyer and Campbell",P005,9,0.0,1
2,SUP003,Acosta Inc,P005,4,0.0,0
3,SUP004,"Evans, Brown and Turner",P002,8,0.0,1
4,SUP005,"Johnson, Wise and Barr",P003,6,0.0,2


In [54]:
agg_monthly_sales = sales.groupby(
[
sales['Sale_Date'].dt.month,
'Product_ID'
]
).agg(
{
'Quantity_Sold':'sum',
'Revenue':'sum'
}
).reset_index()

agg_monthly_sales.head()

,Sale_Date,Product_ID,Quantity_Sold,Revenue
0,1,P001,19896,2120864
1,1,P002,18060,1825528
2,1,P003,22586,2224084
3,1,P004,24849,2514270
4,1,P005,24843,2277116


In [55]:
agg_supplier = suppliers.groupby(
'Supplier_ID'
).agg(
{
'Lead_Time':'mean',
'Order_Frequency':'sum'
}
).reset_index()

agg_supplier.head()

,Supplier_ID,Lead_Time,Order_Frequency
0,SUP001,9.0,0.0
1,SUP002,9.0,0.0
2,SUP003,4.0,0.0
3,SUP004,8.0,0.0
4,SUP005,6.0,0.0


In [56]:
agg_inventory = inventory.groupby(
'Product_ID'
).agg(
{
'Stock_Level':'mean',
'Reorder_Level':'mean'
}
).reset_index()

agg_inventory.head()

,Product_ID,Stock_Level,Reorder_Level
0,P001,231.225000,132.000000
1,P002,247.205128,123.025641
2,P003,225.117647,128.196078
3,P004,221.666667,120.444444
4,P005,195.294118,117.823529


In [57]:
agg_monthly_sales.to_csv(
    'agg_monthly_sales.csv',
    index=False
)

agg_supplier.to_csv(
    'agg_supplier.csv',
    index=False
)

agg_inventory.to_csv(
    'agg_inventory.csv',
    index=False
)

In [58]:
from google.colab import files

files.download('agg_monthly_sales.csv')
files.download('agg_supplier.csv')
files.download('agg_inventory.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>